In [ ]:
import pandas as pd
import json
import torch
df_metric_names = pd.read_json('/kaggle/input/da5401-2025-data-challenge/metric_names.json')
df_train = pd.read_json('/kaggle/input/da5401-2025-data-challenge/train_data.json')
data = np.load('/kaggle/input/da5401-2025-data-challenge/metric_name_embeddings.npy')
df_test = pd.read_json('/kaggle/input/da5401-2025-data-challenge/test_data.json')
df_metric_embd = pd.DataFrame(data)
metric_embeddings = np.load('/kaggle/input/da5401-2025-data-challenge/metric_name_embeddings.npy')
metric_map = {name: torch.tensor(metric_embeddings[i], dtype=torch.float) for i, name in enumerate(df_metric_names[0])}

## EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 6)


# SECTION 1: BASIC DATASET OVERVIEW


def basic_info(df, name="Dataset"):
    """Print basic information about the dataset"""
    print(f"\n{'='*80}")
    print(f"{name.upper()} - BASIC INFORMATION")
    print(f"{'='*80}")
    
    print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nColumn Types:\n{df.dtypes.value_counts()}")
    
    print(f"\n{'-'*80}")
    print("COLUMN DETAILS:")
    print(f"{'-'*80}")
    info_df = pd.DataFrame({
        'Column': df.columns,
        'Type': df.dtypes.values,
        'Non-Null': df.count().values,
        'Null': df.isnull().sum().values,
        'Null %': (df.isnull().sum().values / len(df) * 100).round(2),
        'Unique': df.nunique().values,
        'Unique %': (df.nunique().values / len(df) * 100).round(2)
    })
    print(info_df.to_string(index=False))
    
    # Memory usage
    memory = df.memory_usage(deep=True).sum() / 1024**2
    print(f"\nMemory Usage: {memory:.2f} MB")
    
    return info_df


# SECTION 2: TARGET VARIABLE ANALYSIS (TRAIN ONLY)


def analyze_target(train_df, target_col='score'):
    """Analyze target variable distribution"""
    print(f"\n{'='*80}")
    print(f"TARGET VARIABLE ANALYSIS: '{target_col}'")
    print(f"{'='*80}")
    
    if target_col not in train_df.columns:
        print(f"Warning: '{target_col}' not found in dataset")
        return
    
    target = train_df[target_col]
    
    # Statistics
    print(f"\nTarget Statistics:")
    print(f"  Mean: {target.mean():.4f}")
    print(f"  Median: {target.median():.4f}")
    print(f"  Std: {target.std():.4f}")
    print(f"  Min: {target.min():.4f}")
    print(f"  Max: {target.max():.4f}")
    print(f"  Skewness: {target.skew():.4f}")
    print(f"  Kurtosis: {target.kurtosis():.4f}")
    
    # Distribution
    print(f"\nTarget Distribution:")
    value_counts = target.value_counts().sort_index()
    for val, count in value_counts.items():
        pct = count / len(target) * 100
        print(f"  {val}: {count:5d} ({pct:5.2f}%)")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Histogram
    axes[0].hist(target, bins=30, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel(target_col)
    axes[0].set_ylabel('Frequency')
    axes[0].set_title(f'{target_col} Distribution')
    axes[0].axvline(target.mean(), color='red', linestyle='--', label=f'Mean: {target.mean():.2f}')
    axes[0].axvline(target.median(), color='green', linestyle='--', label=f'Median: {target.median():.2f}')
    axes[0].legend()
    
    # Box plot
    axes[1].boxplot(target)
    axes[1].set_ylabel(target_col)
    axes[1].set_title(f'{target_col} Box Plot')
    axes[1].grid(True, alpha=0.3)
    
    # Count plot for categorical view
    value_counts.plot(kind='bar', ax=axes[2], color='steelblue', edgecolor='black')
    axes[2].set_xlabel(target_col)
    axes[2].set_ylabel('Count')
    axes[2].set_title(f'{target_col} Value Counts')
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Check for imbalance
    print(f"\nClass Imbalance Ratio:")
    max_class = value_counts.max()
    min_class = value_counts.min()
    imbalance_ratio = max_class / min_class
    print(f"  Majority/Minority: {imbalance_ratio:.2f}:1")
    if imbalance_ratio > 3:
        print(f"   WARNING: Significant class imbalance detected!")
        print(f"  Consider: SMOTE, class weights, or stratified sampling")


# SECTION 3: MISSING VALUES ANALYSIS


def analyze_missing(df, name="Dataset"):
    """Analyze missing values"""
    print(f"\n{'='*80}")
    print(f"{name.upper()} - MISSING VALUES ANALYSIS")
    print(f"{'='*80}")
    
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    missing_df = pd.DataFrame({
        'Column': missing.index,
        'Missing': missing.values,
        'Missing %': missing_pct.values
    })
    missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)
    
    if len(missing_df) == 0:
        print("\n No missing values found!")
    else:
        print(f"\nColumns with Missing Values: {len(missing_df)}")
        print(missing_df.to_string(index=False))
        
        # Visualize missing values
        if len(missing_df) > 0:
            plt.figure(figsize=(12, 6))
            missing_df.plot(x='Column', y='Missing %', kind='bar', color='coral', legend=False)
            plt.title(f'{name} - Missing Values Percentage', fontsize=14, fontweight='bold')
            plt.xlabel('Columns')
            plt.ylabel('Missing %')
            plt.xticks(rotation=45, ha='right')
            plt.axhline(y=50, color='red', linestyle='--', label='50% threshold')
            plt.legend()
            plt.tight_layout()
            plt.show()


# SECTION 4: NUMERICAL FEATURES ANALYSIS


def analyze_numerical(df, name="Dataset", target_col=None):
    """Analyze numerical features"""
    print(f"\n{'='*80}")
    print(f"{name.upper()} - NUMERICAL FEATURES ANALYSIS")
    print(f"{'='*80}")
    
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col and target_col in numerical_cols:
        numerical_cols.remove(target_col)
    
    if len(numerical_cols) == 0:
        print("\nNo numerical columns found!")
        return
    
    print(f"\nNumerical Columns: {len(numerical_cols)}")
    
    # Statistical summary
    print(f"\n{'-'*80}")
    print("STATISTICAL SUMMARY:")
    print(f"{'-'*80}")
    print(df[numerical_cols].describe().T)
    
    # Check for outliers using IQR method
    print(f"\n{'-'*80}")
    print("OUTLIER DETECTION (IQR Method):")
    print(f"{'-'*80}")
    
    outlier_summary = []
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
        outlier_summary.append({
            'Column': col,
            'Outliers': len(outliers),
            'Outlier %': f"{(len(outliers)/len(df)*100):.2f}%",
            'Lower Bound': f"{lower_bound:.2f}",
            'Upper Bound': f"{upper_bound:.2f}"
        })
    
    outlier_df = pd.DataFrame(outlier_summary)
    print(outlier_df.to_string(index=False))
    
    # Distribution plots
    n_cols = min(4, len(numerical_cols))
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, col in enumerate(numerical_cols):
        if idx < len(axes):
            df[col].hist(bins=30, ax=axes[idx], edgecolor='black', alpha=0.7)
            axes[idx].set_title(f'{col}\nSkew: {df[col].skew():.2f}', fontsize=10)
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Frequency')
    
    # Hide extra subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f'{name} - Numerical Features Distribution', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    
    # Correlation heatmap
    if len(numerical_cols) > 1:
        plt.figure(figsize=(12, 10))
        corr_matrix = df[numerical_cols].corr()
        
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
                    cmap='coolwarm', center=0, square=True, 
                    linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title(f'{name} - Correlation Heatmap', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # High correlations
        print(f"\n{'-'*80}")
        print("HIGH CORRELATIONS (|r| > 0.7):")
        print(f"{'-'*80}")
        high_corr = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.7:
                    high_corr.append({
                        'Feature 1': corr_matrix.columns[i],
                        'Feature 2': corr_matrix.columns[j],
                        'Correlation': f"{corr_matrix.iloc[i, j]:.4f}"
                    })
        
        if high_corr:
            high_corr_df = pd.DataFrame(high_corr)
            print(high_corr_df.to_string(index=False))
        else:
            print("No high correlations found")


# SECTION 5: CATEGORICAL FEATURES ANALYSIS


def analyze_categorical(df, name="Dataset", target_col=None, max_categories=20):
    """Analyze categorical features"""
    print(f"\n{'='*80}")
    print(f"{name.upper()} - CATEGORICAL FEATURES ANALYSIS")
    print(f"{'='*80}")
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    if len(categorical_cols) == 0:
        print("\nNo categorical columns found!")
        return
    
    print(f"\nCategorical Columns: {len(categorical_cols)}")
    
    # Summary
    cat_summary = []
    for col in categorical_cols:
        cat_summary.append({
            'Column': col,
            'Unique Values': df[col].nunique(),
            'Most Frequent': df[col].mode()[0] if len(df[col].mode()) > 0 else 'N/A',
            'Frequency': df[col].value_counts().iloc[0] if len(df[col]) > 0 else 0,
            'Frequency %': f"{(df[col].value_counts().iloc[0]/len(df)*100):.2f}%" if len(df[col]) > 0 else "0%"
        })
    
    cat_df = pd.DataFrame(cat_summary)
    print(f"\n{cat_df.to_string(index=False)}")
    
    # Visualization
    for col in categorical_cols[:5]:  # Limit to first 5 categorical columns
        unique_vals = df[col].nunique()
        
        if unique_vals > max_categories:
            print(f"\ Skipping '{col}': Too many categories ({unique_vals})")
            continue
        
        plt.figure(figsize=(12, 5))
        
        value_counts = df[col].value_counts().head(max_categories)
        
        plt.subplot(1, 2, 1)
        value_counts.plot(kind='bar', color='teal', edgecolor='black')
        plt.title(f'{col} - Value Counts (Top {min(max_categories, len(value_counts))})')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        
        plt.subplot(1, 2, 2)
        value_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90)
        plt.title(f'{col} - Distribution')
        plt.ylabel('')
        
        plt.tight_layout()
        plt.show()


# SECTION 6: TRAIN vs TEST COMPARISON


def compare_train_test(train_df, test_df, target_col=None):
    """Compare train and test datasets"""
    print(f"\n{'='*80}")
    print("TRAIN vs TEST COMPARISON")
    print(f"{'='*80}")
    
    # Shape comparison
    print(f"\nDataset Shapes:")
    print(f"  Train: {train_df.shape[0]} rows × {train_df.shape[1]} columns")
    print(f"  Test:  {test_df.shape[0]} rows × {test_df.shape[1]} columns")
    
    # Column comparison
    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)
    
    common_cols = train_cols.intersection(test_cols)
    train_only = train_cols - test_cols
    test_only = test_cols - train_cols
    
    print(f"\nColumn Comparison:")
    print(f"  Common columns: {len(common_cols)}")
    if train_only:
        print(f"  Train-only columns: {train_only}")
    if test_only:
        print(f"  Test-only columns: {test_only}")
    
    # Numerical features distribution comparison
    numerical_cols = [col for col in common_cols 
                     if train_df[col].dtype in [np.float64, np.int64, np.float32, np.int32]]
    
    if target_col and target_col in numerical_cols:
        numerical_cols.remove(target_col)
    
    if len(numerical_cols) > 0:
        print(f"\n{'-'*80}")
        print("NUMERICAL FEATURES - STATISTICAL COMPARISON:")
        print(f"{'-'*80}")
        
        comparison = []
        for col in numerical_cols[:10]:  # Limit to first 10
            comparison.append({
                'Feature': col,
                'Train Mean': f"{train_df[col].mean():.4f}",
                'Test Mean': f"{test_df[col].mean():.4f}",
                'Train Std': f"{train_df[col].std():.4f}",
                'Test Std': f"{test_df[col].std():.4f}",
                'KS-Stat': f"{stats.ks_2samp(train_df[col].dropna(), test_df[col].dropna())[0]:.4f}"
            })
        
        comp_df = pd.DataFrame(comparison)
        print(comp_df.to_string(index=False))
        
        # Distribution comparison plots
        n_plots = min(6, len(numerical_cols))
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.flatten()
        
        for idx, col in enumerate(numerical_cols[:n_plots]):
            axes[idx].hist(train_df[col].dropna(), bins=30, alpha=0.5, label='Train', color='blue', density=True)
            axes[idx].hist(test_df[col].dropna(), bins=30, alpha=0.5, label='Test', color='orange', density=True)
            axes[idx].set_title(f'{col}')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Density')
            axes[idx].legend()
        
        plt.suptitle('Train vs Test - Distribution Comparison', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()


# SECTION 7: FEATURE ENGINEERING SUGGESTIONS


def suggest_feature_engineering(train_df, test_df=None, target_col=None):
    """Suggest feature engineering opportunities"""
    print(f"\n{'='*80}")
    print("FEATURE ENGINEERING SUGGESTIONS")
    print(f"{'='*80}")
    
    suggestions = []
    
    # Check for highly skewed features
    numerical_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col and target_col in numerical_cols:
        numerical_cols.remove(target_col)
    
    print("\n1. SKEWED FEATURES (Consider log/sqrt transformation):")
    for col in numerical_cols:
        skew = train_df[col].skew()
        if abs(skew) > 1:
            print(f"   • {col}: skewness = {skew:.2f}")
            suggestions.append(f"Apply log/sqrt transform to '{col}'")
    
    # Check for high cardinality categorical features
    categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    if categorical_cols:
        print("\n2. HIGH CARDINALITY CATEGORICAL (Consider encoding):")
        for col in categorical_cols:
            unique = train_df[col].nunique()
            if unique > 10:
                print(f"   • {col}: {unique} unique values")
                suggestions.append(f"Apply target encoding or frequency encoding to '{col}'")
    
    # Check for potential interaction features
    if len(numerical_cols) > 1:
        print("\n3. POTENTIAL INTERACTIONS (Consider creating):")
        print("   • Polynomial features from highly correlated pairs")
        print("   • Ratio features (e.g., feature1/feature2)")
        suggestions.append("Create interaction features between correlated variables")
    
    # Check for date columns
    date_cols = train_df.select_dtypes(include=['datetime64']).columns.tolist()
    if date_cols:
        print("\n4. DATE FEATURES (Extract temporal features):")
        for col in date_cols:
            print(f"   • {col}: Extract year, month, day, day_of_week, etc.")
            suggestions.append(f"Extract temporal features from '{col}'")
    
    print("\n5. OTHER SUGGESTIONS:")
    print("   • Create aggregated features (mean, sum, count by groups)")
    print("   • Engineer domain-specific features")
    print("   • Consider PCA for dimensionality reduction")
    
    return suggestions


# MAIN EXECUTION


def full_eda(train_df, test_df=None, target_col='score'):
   
    
    print("\n" + "="*80)
    print(" "*25 + "EXPLORATORY DATA ANALYSIS")
    print("="*80)
    
    # Section 1: Basic Info
    train_info = basic_info(train_df, "TRAIN")
    if test_df is not None:
        test_info = basic_info(test_df, "TEST")
    
    # Section 2: Target Analysis (Train only)
    if target_col in train_df.columns:
        analyze_target(train_df, target_col)
    
    # Section 3: Missing Values
    analyze_missing(train_df, "TRAIN")
    if test_df is not None:
        analyze_missing(test_df, "TEST")
    
    # Section 4: Numerical Features
    analyze_numerical(train_df, "TRAIN", target_col)
    if test_df is not None:
        analyze_numerical(test_df, "TEST", target_col)
    
    # Section 5: Categorical Features
    analyze_categorical(train_df, "TRAIN", target_col)
    if test_df is not None:
        analyze_categorical(test_df, "TEST", target_col)
    
    # Section 6: Train vs Test Comparison
    if test_df is not None:
        compare_train_test(train_df, test_df, target_col)
    
    # Section 7: Feature Engineering Suggestions
    suggest_feature_engineering(train_df, test_df, target_col)
    
    print("\n" + "="*80)
    print(" "*30 + "EDA COMPLETE!")
    print("="*80)




# Run the complete EDA
full_eda(train_df, test_df, target_col='score')



## Embeddings Creation

In [ ]:
import re

def preprocess_text(text):
 
    # 1. Handle non-string inputs (like NaN)
    if not isinstance(text, str):
        return ""
        
    # 2. Lowercase the text
    text = text.lower()
    
    # 3. Remove URLs
    text = re.sub(r'https://\S+|www\.\S+|http\S+', '', text, flags=re.MULTILINE)
    
    # 4. Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 5. Remove basic HTML tags (just in case)
    text = re.sub(r'<.*?>', '', text)
    
    # 6. Replace newlines and tabs with a single space
    text = re.sub(r'[\n\t]', ' ', text)
    
    # 7. Collapse multiple spaces into one and strip leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:

from sentence_transformers import SentenceTransformer
import torch
def preprocess_and_engineer_features_labse(train_df, test_df, metric_names, metric_embeddings):

    
    # 1. Clean Text 
    print("Cleaning text data")
    for col in TEXT_COLS: # TEXT_COLS = ['user_prompt', 'response', 'system_prompt']
        train_df[col] = train_df[col].apply(preprocess_text)
        test_df[col] = test_df[col].apply(preprocess_text)
        
    # 2. Create Metric Embedding Map 
    metric_map = dict(zip(metric_names, metric_embeddings))
    
    # 3. Load LaBSE Model (EXACTLY like your working code) 
    print("Loading embedding model (LaBSE)")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    from sentence_transformers import SentenceTransformer
    SentenceTransformer.cache_root = '/tmp/sentence-transformers'  # Change cache directory
    embedding_model = SentenceTransformer("sentence-transformers/LaBSE")
    # USE THE EXACT SAME APPROACH THAT WORKED 
     # = SentenceTransformer('l3cube-pune/indic-sentence-similarity-sbert')
    # embedding_model = SentenceTransformer(model_name)  # No extra parameters!
    
    x_prompt = []
    x_response = []
    x_metric = []
    x_prompt_test = []
    x_response_test = []
    x_metric_test = []
    
    for df in [train_df, test_df]:
        # A. Get Metric Embeddings (The "Key" from Gemma)
        # Shape: (N, 768)
        metric_emb = np.stack(df['metric_name'].map(metric_map).values)
        
        # B. Create Pair Embeddings (The "Content" from LaBSE)
        print(f"Concatenating *cleaned* text for {'train' if 'score' in df.columns else 'test'}")
        prompt_text = "prompt: " + df['user_prompt']
        response_text = "response: " + df['response']
        
        print(f"Encoding text with LaBSE (this will be slower)")
        # Use the exact same encoding approach that worked
        prompt_emb = embedding_model.encode(
            prompt_text.tolist(), 
            show_progress_bar=True, 
            batch_size=64
        )
        response_emb = embedding_model.encode(
            response_text.tolist(), 
            show_progress_bar=True, 
            batch_size=64
        )
        
        
        # C. Combine all features
        # LaBSE outputs a 768-dim vector, same as Gemma
        # Shape: (N, 768 + 768) = (N, 1536)
        final_features = np.hstack([metric_emb, prompt_emb, response_emb])
        
        if 'score' in df.columns:
            x_prompt.append(prompt_emb)
            x_response.append(response_emb)
            x_metric.append(metric_emb)
            
        else:
            x_prompt_test.append(prompt_emb)
            x_response_test.append(response_emb)
            x_metric_test.append(metric_emb)
    
    X_prompt = np.vstack(x_prompt)
    X_response = np.vstack(x_response)
    X_metric = np.vstack(x_metric)
    X_prompt_test = np.vstack(x_prompt_test)
    X_response_test = np.vstack(x_response_test)
    X_metric_test = np.vstack(x_metric_test)
    
    y = train_df[TARGET_COL].values
    groups = train_df['metric_name'].values
    test_ids = test_df['ID'].values
    
    print(f"Feature engineering complete.")
    print(f"X_prompt shape: {X_prompt.shape}") # (N, 1536)
    print(f"X_response shape: {X_response.shape}")
    print(f"X_metric shape: {X_metric.shape}")
    print(f"y shape: {y.shape}")
    # print(f"X_test shape: {X_test.shape}")
    
    return X_prompt, X_response, X_metric, X_prompt_test, X_response_test, X_metric_test, groups, test_ids

# Run NEW Feature Engineering 
X_prompt, X_response, X_metric, X_prompt_test, X_response_test, X_metric_test, groups, test_ids = preprocess_and_engineer_features_labse(
    train_df, test_df, metric_names, metric_embeddings
)

In [ ]:
import numpy as np
np.save('/kaggle/input/embeddings-labse-v2/X_prompt.npy', X_prompt)
np.save('/kaggle/input/embeddings-labse-v2/X_response.npy', X_response)
np.save('/kaggle/input/embeddings-labse-v2/X_metric.npy', X_metric)
np.save('/kaggle/input/embeddings-labse-v2/X_prompt_test.npy', X_prompt_test)
np.save('/kaggle/input/embeddings-labse-v2/X_response_test.npy', X_response_test)
np.save('/kaggle/input/embeddings-labse-v2/X_metric_test.npy', X_metric_test)
np.save('/kaggle/input/embeddings-labse-v2/y.npy', y)
np.save('/kaggle/input/embeddings-labse-v2/groups.npy', groups)
np.save('/kaggle/input/embeddings-labse-v2/test_ids.npy', test_ids)

In [ ]:

from sentence_transformers import SentenceTransformer
import torch
def preprocess_and_engineer_features_indic(train_df, test_df, metric_names, metric_embeddings):

    
    # 1. Clean Text 
    print("Cleaning text data")
    for col in TEXT_COLS: # TEXT_COLS = ['user_prompt', 'response', 'system_prompt']
        train_df[col] = train_df[col].apply(preprocess_text)
        test_df[col] = test_df[col].apply(preprocess_text)
        
    # 2. Create Metric Embedding Map 
    metric_map = dict(zip(metric_names, metric_embeddings))
    
    # 3. Load LaBSE Model (EXACTLY like your working code) 
    print("Loading embedding model (LaBSE)")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    from sentence_transformers import SentenceTransformer
    SentenceTransformer.cache_root = '/tmp/sentence-transformers'  # Change cache directory
    embedding_model = SentenceTransformer("l3cube-pune/indic-sentence-similarity-sbert")
    # USE THE EXACT SAME APPROACH THAT WORKED 
     # = SentenceTransformer('l3cube-pune/indic-sentence-similarity-sbert')
    # embedding_model = SentenceTransformer(model_name)  # No extra parameters!
    
    x_prompt = []
    x_response = []
    x_metric = []
    x_prompt_test = []
    x_response_test = []
    x_metric_test = []
    
    for df in [train_df, test_df]:
        # A. Get Metric Embeddings (The "Key" from Gemma)
        # Shape: (N, 768)
        metric_emb = np.stack(df['metric_name'].map(metric_map).values)
        
        # B. Create Pair Embeddings (The "Content" from LaBSE)
        print(f"Concatenating *cleaned* text for {'train' if 'score' in df.columns else 'test'}")
        prompt_text = "prompt: " + df['user_prompt']
        response_text = "response: " + df['response']
        
        print(f"Encoding text with LaBSE (this will be slower)")
        # Use the exact same encoding approach that worked
        prompt_emb = embedding_model.encode(
            prompt_text.tolist(), 
            show_progress_bar=True, 
            batch_size=64
        )
        response_emb = embedding_model.encode(
            response_text.tolist(), 
            show_progress_bar=True, 
            batch_size=64
        )
        
        
        # C. Combine all features
        # LaBSE outputs a 768-dim vector, same as Gemma
        # Shape: (N, 768 + 768) = (N, 1536)
        final_features = np.hstack([metric_emb, prompt_emb, response_emb])
        
        if 'score' in df.columns:
            x_prompt.append(prompt_emb)
            x_response.append(response_emb)
            x_metric.append(metric_emb)
            
        else:
            x_prompt_test.append(prompt_emb)
            x_response_test.append(response_emb)
            x_metric_test.append(metric_emb)
    
    X_prompt = np.vstack(x_prompt)
    X_response = np.vstack(x_response)
    X_metric = np.vstack(x_metric)
    X_prompt_test = np.vstack(x_prompt_test)
    X_response_test = np.vstack(x_response_test)
    X_metric_test = np.vstack(x_metric_test)
    
    y = train_df[TARGET_COL].values
    groups = train_df['metric_name'].values
    test_ids = test_df['ID'].values
    
    print(f"Feature engineering complete.")
    print(f"X_prompt shape: {X_prompt.shape}") # (N, 1536)
    print(f"X_response shape: {X_response.shape}")
    print(f"X_metric shape: {X_metric.shape}")
    print(f"y shape: {y.shape}")
    # print(f"X_test shape: {X_test.shape}")
    
    return X_prompt, X_response, X_metric, X_prompt_test, X_response_test, X_metric_test, groups, test_ids

# Run NEW Feature Engineering 
X_prompt, X_response, X_metric, X_prompt_test, X_response_test, X_metric_test, groups, test_ids = preprocess_and_engineer_features_indic(
    train_df, test_df, metric_names, metric_embeddings
)

In [ ]:
import numpy as np
np.save('/kaggle/input/indic-bert-pune/X_prompt.npy', X_prompt)
np.save('/kaggle/input/indic-bert-pune/X_response.npy', X_response)
np.save('/kaggle/input/indic-bert-pune/X_metric.npy', X_metric)
np.save('/kaggle/input/indic-bert-pune/X_prompt_test.npy', X_prompt_test)
np.save('/kaggle/input/indic-bert-pune/X_response_test.npy', X_response_test)
np.save('/kaggle/input/indic-bert-pune/X_metric_test.npy', X_metric_test)
# np.save('y.npy', y)
np.save('/kaggle/input/indic-bert-pune/groups.npy', groups)
np.save('/kaggle/input/indic-bert-pune/test_ids.npy', test_ids)

In [ ]:
#generate_system_embeddings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import os
import warnings

warnings.filterwarnings('ignore')

#  Configuration 
TRAIN_FILE = '/kaggle/input/da5401-2025-data-challenge/train_data.json'
TEST_FILE = '/kaggle/input/da5401-2025-data-challenge/test_data.json'
SAVE_PATH = "/kaggle/working/system_embeddings/" # All output files go here
BATCH_SIZE = 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# List of all transformer models you want to use for the system prompt
MODEL_LIST = [
    'sentence-transformers/LaBSE'
]

#  Helper Function: Mean Pooling 
def mean_pooling(model_output, attention_mask):
    """Mean pooling for sentence embeddings"""
    token_embeddings = model_output[0] # First element of model_output
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

#  Main Embedding Generation Function 
def generate_embeddings(texts, model_name, batch_size, device):
    """
    Generates normalized embeddings for a list of texts,
    handling missing values.
    """
    print(f"  Loading model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    
    # Handle missing values: Replace both NaN/None and empty strings
    # with a single, special token.
    processed_texts = pd.Series(texts).fillna("[MISSING]").replace("", "[MISSING]").tolist()
    
    all_embeddings = []
    # Use tqdm for a progress bar
    for i in tqdm(range(0, len(processed_texts), batch_size), desc="  Batches"):
        batch = processed_texts[i:i+batch_size]
        
        # Tokenize batch
        inputs = tokenizer(
            batch, 
            padding=True, 
            truncation=True, 
            max_length=256, # System prompts are usually short
            return_tensors="pt"
        ).to(device)
        
        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = mean_pooling(outputs, inputs['attention_mask'])
            # Normalize embeddings for stable similarity calculations
            embeddings = F.normalize(embeddings, p=2, dim=1) 
        
        all_embeddings.append(embeddings.cpu().numpy())
        
    return np.vstack(all_embeddings)

#  Main Execution 
if __name__ == "__main__":
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)
        
    print(f"Loading data from {TRAIN_FILE} and {TEST_FILE}...")
    try:
        train_df = pd.read_json(TRAIN_FILE)
        test_df = pd.read_json(TEST_FILE)
    except Exception as e:
        print(f"ERROR: Could not load data files. {e}")
        exit()

    train_prompts = train_df['system_prompt'].values
    test_prompts = test_df['system_prompt'].values
    
    #  1. Generate embeddings for each model 
    for model_name in MODEL_LIST:
        print(f"\nProcessing model: {model_name}")
        # Create a clean filename
        model_key = model_name.split('/')[-1].replace('-', '_').lower()
        
        
        # Create Train Embeddings
        train_embeds = generate_embeddings(train_prompts, model_name, BATCH_SIZE, DEVICE)
        train_save_file = os.path.join(SAVE_PATH, f"X_system_{model_key}_train.npy")
        np.save(train_save_file, train_embeds)
        print(f"   Saved train embeddings: {train_save_file} (Shape: {train_embeds.shape})")
        
        # Create Test Embeddings
        test_embeds = generate_embeddings(test_prompts, model_name, BATCH_SIZE, DEVICE)
        test_save_file = os.path.join(SAVE_PATH, f"X_system_{model_key}_test.npy")
        np.save(test_save_file, test_embeds)
        print(f"   Saved test embeddings: {test_save_file} (Shape: {test_embeds.shape})")
       

    #  2. Generate the binary "missing" indicator feature (CRITICAL) 
    print("\nGenerating missing indicator features...")
    # .isna() catches None and np.nan.
    train_missing = pd.Series(train_prompts).isna().astype(int).values.reshape(-1, 1)
    test_missing = pd.Series(test_prompts).isna().astype(int).values.reshape(-1, 1)
    
    np.save(os.path.join(SAVE_PATH, "/kaggle/input/embeddings-labse-v2/system_missing_train.npy"), train_missing)
    np.save(os.path.join(SAVE_PATH, "/kaggle/input/embeddings-labse-v2/system_missing_test.npy"), test_missing)
    print("  Saved missing indicators (system_missing_train.npy, system_missing_test.npy)")
    


## Actual Model

In [ ]:
import os, gc, sys, time, math, random
from collections import Counter
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from numpy.linalg import norm


# CONFIG

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

EMBED_PATH = "/kaggle/input/embeddings-labse-v2/"
INDIC_PATH = "/kaggle/input/indic-bert-pune/"
SYSTEM_EMBED_PATH = "/kaggle/input/embeddings-labse-v2/"

N_NEG_SAMPLES = 2000  
NUM_DOMAINS = 5       # 5 Experts
ISO_CONTAMINATION = 0.03
PCA_COMPONENTS = 32
MINIBATCH_KMEANS_BATCH = 64
KFOLD = 10
OPTUNA_TRIALS = 50    # 50 trials per model per domain

# Fixed Params for Classifier (Stabilizer)
LGB_CLS_PARAMS = dict(
    objective='multiclass', metric='multi_logloss', n_estimators=800,
    learning_rate=0.02, num_leaves=32, random_state=SEED, n_jobs=-1, verbose=-1
)

USE_TEMP_CALIBRATION = True


# HELPERS & FACTORY

def now(): return time.strftime("%H:%M:%S")
def memprint(msg=""): print(f"[{now()}] {msg}")
def safe_load(path): return np.load(path, allow_pickle=True) if os.path.exists(path) else None
optuna.logging.set_verbosity(optuna.logging.ERROR)

class FeatureFactory:
    def __init__(self): self.eps = 1e-8
    def l2norm(self, a): return np.linalg.norm(a, axis=1, keepdims=True) + self.eps
    def cosine_sim(self, A, B):
        num = np.sum(A * B, axis=1)
        den = (norm(A, axis=1) * norm(B, axis=1)) + self.eps
        return num / den
    def euclidean_dist(self, A, B): return norm(A - B, axis=1)
    def scalar_proj(self, A, B): return np.sum(A * B, axis=1) / (norm(B, axis=1) + self.eps)

    def create_features(self, Xp, Xr, Xm):
        sim_rm = self.cosine_sim(Xr, Xm); sim_pm = self.cosine_sim(Xp, Xm); sim_pr = self.cosine_sim(Xp, Xr)
        dist_rm = self.euclidean_dist(Xr, Xm); dist_pm = self.euclidean_dist(Xp, Xm); dist_pr = self.euclidean_dist(Xp, Xr)
        proj_r_on_m = self.scalar_proj(Xr, Xm); delta_rp = Xr - Xp; proj_delta_on_m = self.scalar_proj(delta_rp, Xm)
        mag_r = norm(Xr, axis=1); mag_m = norm(Xm, axis=1)
        var_r = np.var(Xr, axis=1); var_m = np.var(Xm, axis=1)
        ratio_align = sim_rm / (sim_pm + self.eps); attn = np.sum(Xr * Xm, axis=1)
        return np.column_stack([
            sim_rm, sim_pm, sim_pr, dist_rm, dist_pm, dist_pr,
            proj_r_on_m, proj_delta_on_m, mag_r, mag_m, var_r, var_m, ratio_align, attn
        ]).astype(np.float32)

factory = FeatureFactory()

def proxy_system_interactions(Xs_pca, Xp, Xr, Xm, k=8):
    k = min(k, Xp.shape[1], Xs_pca.shape[1])
    def proj_mean(A): return A[:, :k].mean(axis=1, keepdims=True)
    sx = Xs_pca[:, :k]
    p_proxy = proj_mean(Xp); r_proxy = proj_mean(Xr); m_proxy = proj_mean(Xm)
    sx_mean = sx.mean(axis=1, keepdims=True)
    sx_norm = sx_mean / (np.linalg.norm(sx_mean, axis=1, keepdims=True) + 1e-8)
    p_norm = p_proxy / (np.linalg.norm(p_proxy, axis=1, keepdims=True) + 1e-8)
    r_norm = r_proxy / (np.linalg.norm(r_proxy, axis=1, keepdims=True) + 1e-8)
    m_norm = m_proxy / (np.linalg.norm(m_proxy, axis=1, keepdims=True) + 1e-8)
    return np.hstack([
        (sx_norm * p_norm).sum(axis=1, keepdims=True), (sx_norm * r_norm).sum(axis=1, keepdims=True), (sx_norm * m_norm).sum(axis=1, keepdims=True),
        np.abs(sx_mean - p_proxy), np.abs(sx_mean - r_proxy), np.abs(sx_mean - m_proxy)
    ]).astype(np.float32)


# LOAD DATA

memprint("Loading input arrays")
try:
    Xp_labse = safe_load(f"{EMBED_PATH}X_prompt.npy"); Xr_labse = safe_load(f"{EMBED_PATH}X_response.npy"); Xm_labse = safe_load(f"{EMBED_PATH}X_metric.npy")
    Xp_labse_te = safe_load(f"{EMBED_PATH}X_prompt_test.npy"); Xr_labse_te = safe_load(f"{EMBED_PATH}X_response_test.npy"); Xm_labse_te = safe_load(f"{EMBED_PATH}X_metric_test.npy")
    Xp_indic = safe_load(f"{INDIC_PATH}X_prompt.npy"); Xr_indic = safe_load(f"{INDIC_PATH}X_response.npy"); Xm_indic = safe_load(f"{INDIC_PATH}X_metric.npy")
    Xp_indic_te = safe_load(f"{INDIC_PATH}X_prompt_test.npy"); Xr_indic_te = safe_load(f"{INDIC_PATH}X_response_test.npy"); Xm_indic_te = safe_load(f"{INDIC_PATH}X_metric_test.npy")
    Xs_train = safe_load(f"{SYSTEM_EMBED_PATH}X_system_labse_train.npy"); Xs_test = safe_load(f"{SYSTEM_EMBED_PATH}X_system_labse_test.npy")
    miss_tr = safe_load(f"{SYSTEM_EMBED_PATH}system_missing_train.npy"); miss_te = safe_load(f"{SYSTEM_EMBED_PATH}system_missing_test.npy")
    groups = safe_load(f"{EMBED_PATH}groups.npy"); y = safe_load(f"{EMBED_PATH}y.npy"); test_ids = safe_load(f"{EMBED_PATH}test_ids.npy")
    metric_pool_labse = safe_load(f"{EMBED_PATH}metric_pool_labse.npy"); metric_pool_indic = safe_load(f"{INDIC_PATH}metric_pool_indic.npy")
except: pass

if y is None:
    memprint("Using synthetic data.")
    N_TRAIN, N_TEST, DIM = 5000, 3638, 768
    Xp_labse = np.random.randn(N_TRAIN, DIM).astype(np.float32); Xp_labse_te = np.random.randn(N_TEST, DIM).astype(np.float32)
    Xr_labse = np.random.randn(N_TRAIN, DIM).astype(np.float32); Xr_labse_te = np.random.randn(N_TEST, DIM).astype(np.float32)
    Xm_labse = np.random.randn(N_TRAIN, DIM).astype(np.float32); Xm_labse_te = np.random.randn(N_TEST, DIM).astype(np.float32)
    Xp_indic = np.random.randn(N_TRAIN, DIM).astype(np.float32); Xp_indic_te = np.random.randn(N_TEST, DIM).astype(np.float32)
    Xr_indic = np.random.randn(N_TRAIN, DIM).astype(np.float32); Xr_indic_te = np.random.randn(N_TEST, DIM).astype(np.float32)
    Xm_indic = np.random.randn(N_TRAIN, DIM).astype(np.float32); Xm_indic_te = np.random.randn(N_TEST, DIM).astype(np.float32)
    Xs_train = np.random.randn(N_TRAIN, PCA_COMPONENTS).astype(np.float32); Xs_test = np.random.randn(N_TEST, PCA_COMPONENTS).astype(np.float32)
    miss_tr = np.random.randint(0,2,(N_TRAIN,1)).astype(np.float32); miss_te = np.random.randint(0,2,(N_TEST,1)).astype(np.float32)
    y = np.clip(np.random.randn(N_TRAIN) * 1.5 + 9.0, 0, 10).astype(np.float32)
    groups = np.arange(N_TRAIN) % 145
    test_ids = np.arange(N_TEST)
    metric_pool_labse = Xm_labse[:145].copy(); metric_pool_indic = Xm_indic[:145].copy()

for name in ["Xp_labse","Xr_labse","Xm_labse","Xp_labse_te","Xr_labse_te","Xm_labse_te",
             "Xp_indic","Xr_indic","Xm_indic","Xp_indic_te","Xr_indic_te","Xm_indic_te",
             "Xs_train","Xs_test","miss_tr","miss_te"]:
    arr = locals().get(name)
    if arr is not None and arr.dtype != np.float32: locals()[name] = arr.astype(np.float32)

if miss_tr is None: miss_tr = np.zeros((len(y), 1), dtype=np.float32)
if miss_te is None: miss_te = np.zeros((len(test_ids), 1), dtype=np.float32)
miss_tr = miss_tr.reshape(-1, 1); miss_te = miss_te.reshape(-1, 1)


# STAGE 1: Feature Factory (NO DIFF)

memprint("Feature Engineering")
f_l = factory.create_features(Xp_labse, Xr_labse, Xm_labse) # 14 Feats
f_i = factory.create_features(Xp_indic, Xr_indic, Xm_indic) # 14 Feats

# CHANGE: No f_diff 
X_sim = np.hstack([f_l, f_i]) # 28 Feats

f_l_te = factory.create_features(Xp_labse_te, Xr_labse_te, Xm_labse_te)
f_i_te = factory.create_features(Xp_indic_te, Xr_indic_te, Xm_indic_te)
X_sim_te = np.hstack([f_l_te, f_i_te]) # 28 Feats

iso = IsolationForest(contamination=ISO_CONTAMINATION, random_state=SEED).fit(X_sim)
anom_tr = iso.decision_function(X_sim).reshape(-1,1); anom_te = iso.decision_function(X_sim_te).reshape(-1,1)

pca = PCA(n_components=PCA_COMPONENTS, random_state=SEED)
if Xs_train.shape[1] > PCA_COMPONENTS:
    Xs_tr = pca.fit_transform(Xs_train); Xs_te = pca.transform(Xs_test)
else: Xs_tr, Xs_te = Xs_train, Xs_test

sys_tr = proxy_system_interactions(Xs_tr, Xp_labse, Xr_labse, Xm_labse)
sys_te = proxy_system_interactions(Xs_te, Xp_labse_te, Xr_labse_te, Xm_labse_te)

X_train = np.hstack([X_sim, anom_tr, sys_tr, Xs_tr, miss_tr]).astype(np.float32)
X_test = np.hstack([X_sim_te, anom_te, sys_te, Xs_te, miss_te]).astype(np.float32)
memprint(f"Train Features: {X_train.shape}")

del X_sim, X_sim_te, f_l, f_i, f_l_te, f_i_te, anom_tr, anom_te; gc.collect()


# STAGE 2: Augmentation (NO DIFF)

memprint(f"Augmenting (N={N_NEG_SAMPLES})")
if metric_pool_labse is None: metric_pool_labse = Xm_labse[:145]
if metric_pool_indic is None: metric_pool_indic = Xm_indic[:145]

def gen_neg(Xp_l, Xr_l, Xm_pool_l, Xp_i, Xr_i, Xm_pool_i, Xs_p, X_m, y_tr, n):
    idx = np.where(y_tr >= 8.0)[0]
    rng = np.random.RandomState(SEED)
    feats, labs = [], []
    for _ in range(n):
        b = rng.choice(idx); m = rng.randint(0, len(Xm_pool_l))
        fl = factory.create_features(Xp_l[b:b+1], Xr_l[b:b+1], Xm_pool_l[m:m+1])
        fi = factory.create_features(Xp_i[b:b+1], Xr_i[b:b+1], Xm_pool_i[m:m+1])
        
        # CHANGE: No fd (difference) 
        sim = np.hstack([fl, fi])
        
        anom = iso.decision_function(sim).reshape(1,1)
        sys = proxy_system_interactions(Xs_p[b:b+1], Xp_l[b:b+1], Xr_l[b:b+1], Xm_pool_l[m:m+1])
        full = np.hstack([sim, anom, sys, Xs_p[b:b+1], X_m[b:b+1]]).squeeze(0)
        if full.shape[0] != X_train.shape[1]: full = np.pad(full, (0, max(0, X_train.shape[1]-len(full))))[:X_train.shape[1]]
        feats.append(full); labs.append(rng.choice([1.0, 1.5, 2.0]))
    return np.vstack(feats).astype(np.float32), np.array(labs)

X_aug, y_aug = gen_neg(Xp_labse, Xr_labse, metric_pool_labse, Xp_indic, Xr_indic, metric_pool_indic, Xs_tr, miss_tr, y, N_NEG_SAMPLES)

# Mapping
nn = NearestNeighbors(n_neighbors=1).fit(metric_pool_labse)
def get_map(X): 
    idx = np.zeros(len(X), dtype=int)
    for i in range(0, len(X), 2048): idx[i:i+2048] = nn.kneighbors(X[i:i+2048])[1].ravel()
    return idx
row_map = get_map(Xm_labse); row_map_te = get_map(Xm_labse_te)

del Xp_labse, Xr_labse, Xm_labse, Xp_indic, Xr_indic, Xm_indic; gc.collect()

# Clustering
memprint("Clustering")
km = MiniBatchKMeans(n_clusters=NUM_DOMAINS, random_state=SEED, batch_size=64, n_init=10).fit(metric_pool_labse)
dist = km.transform(metric_pool_labse); w_pool = np.exp(-dist*5); w_pool /= w_pool.sum(axis=1, keepdims=True)
w_orig = w_pool[row_map]; w_aug = np.full((len(y_aug), NUM_DOMAINS), 1/NUM_DOMAINS)

if len(X_aug)>0:
    X_full = np.vstack([X_train, X_aug]); y_full = np.concatenate([y, y_aug])
    w_full = np.vstack([w_orig, w_aug])
else: X_full, y_full, w_full = X_train, y, w_orig

y_class = np.clip(np.round(y_full).astype(int), 0, 10)
classes = np.arange(11, dtype=float)


# STAGE 4: LOCAL TUNING

cv_tune = KFold(n_splits=3, shuffle=True, random_state=SEED)

def tune_lgbm(X, y, w):
    def obj(trial):
        p = {'objective':'regression_l1','metric':'mae','verbosity':-1,'n_jobs':-1,'random_state':SEED,
             'n_estimators':500,
             'learning_rate':trial.suggest_float('learning_rate',0.01,0.1),
             'num_leaves':trial.suggest_int('num_leaves',15,63),
             'reg_alpha':trial.suggest_float('reg_alpha',1e-8,1.0,log=True)}
        tr, val = next(iter(cv_tune.split(X, y)))
        m = lgb.LGBMRegressor(**p)
        m.fit(X[tr], y[tr], sample_weight=w[tr], eval_set=[(X[val], y[val])], callbacks=[lgb.early_stopping(30,verbose=False)])
        return np.sqrt(mean_squared_error(y[val], m.predict(X[val])))
    study = optuna.create_study(direction='minimize'); study.optimize(obj, n_trials=OPTUNA_TRIALS)
    bp = study.best_params; bp.update({'objective':'regression_l1','n_estimators':1200,'n_jobs':-1,'random_state':SEED,'verbose':-1})
    return bp

def tune_xgb(X, y, w):
    def obj(trial):
        p = {'objective':'reg:absoluteerror','n_jobs':-1,'random_state':SEED,'tree_method':'hist',
             'n_estimators':500,
             'learning_rate':trial.suggest_float('learning_rate',0.01,0.1),
             'max_depth':trial.suggest_int('max_depth',3,8),
             'reg_alpha':trial.suggest_float('reg_alpha',1e-8,1.0,log=True)}
        tr, val = next(iter(cv_tune.split(X, y)))
        m = xgb.XGBRegressor(**p)
        m.fit(X[tr], y[tr], sample_weight=w[tr], eval_set=[(X[val], y[val])], verbose=False)
        return np.sqrt(mean_squared_error(y[val], m.predict(X[val])))
    study = optuna.create_study(direction='minimize'); study.optimize(obj, n_trials=OPTUNA_TRIALS)
    bp = study.best_params; bp.update({'objective':'reg:absoluteerror','n_estimators':1000,'n_jobs':-1,'random_state':SEED,'tree_method':'hist'})
    return bp

def tune_cat(X, y, w):
    def obj(trial):
        p = {'loss_function':'MAE','random_seed':SEED,'verbose':0,'allow_writing_files':False,
             'iterations':500,
             'learning_rate':trial.suggest_float('learning_rate',0.01,0.1),
             'depth':trial.suggest_int('depth',4,8),
             'l2_leaf_reg':trial.suggest_float('l2_leaf_reg',1e-3,10.0,log=True),
             'subsample':trial.suggest_float('subsample',0.5,1.0)}
        tr, val = next(iter(cv_tune.split(X, y)))
        tp = Pool(X[tr], y[tr], weight=w[tr]); vp = Pool(X[val], y[val], weight=w[val])
        m = CatBoostRegressor(**p)
        m.fit(tp, eval_set=vp, early_stopping_rounds=30)
        return np.sqrt(mean_squared_error(y[val], m.predict(X[val])))
    study = optuna.create_study(direction='minimize'); study.optimize(obj, n_trials=OPTUNA_TRIALS)
    bp = study.best_params; bp.update({'loss_function':'MAE','iterations':1000,'random_seed':SEED,'verbose':0,'allow_writing_files':False})
    return bp


# STAGE 5: TRAIN HETEROGENEOUS EXPERTS

cv = KFold(n_splits=KFOLD, shuffle=True, random_state=SEED)
oof_expert = np.zeros((len(X_full), NUM_DOMAINS), dtype=np.float32)
test_expert = np.zeros((len(X_test), NUM_DOMAINS), dtype=np.float32)

memprint("Training Heterogeneous Committees (Locally Tuned)")
for d in range(NUM_DOMAINS):
    memprint(f"  Domain {d}")
    wt = w_full[:, d].astype(float)
    if len(y_aug)>0: wt[-len(y_aug):] *= 2.0
    
    idx = np.random.choice(len(y_full), min(1500, len(y_full)), p=wt/wt.sum(), replace=False)
    memprint("    Tuning")
    BP_LGB = tune_lgbm(X_full[idx], y_full[idx], wt[idx])
    BP_XGB = tune_xgb(X_full[idx], y_full[idx], wt[idx])
    BP_CAT = tune_cat(X_full[idx], y_full[idx], wt[idx])
    
    fold_test = np.zeros(len(X_test), dtype=np.float32)
    for tr, val in cv.split(X_full, y_full):
        xt, xv = X_full[tr], X_full[val]; yt, yv = y_full[tr], y_full[val]; w = wt[tr]
        
        # 1. LGBM Reg
        m1 = lgb.LGBMRegressor(**BP_LGB)
        m1.fit(xt, yt, sample_weight=w, eval_set=[(xv, yv)], eval_metric='mae', callbacks=[lgb.early_stopping(50, verbose=False)])
        p1 = m1.predict(xv); t1 = m1.predict(X_test)
        
        # 2. XGB Reg
        m2 = xgb.XGBRegressor(**BP_XGB)
        m2.fit(xt, yt, sample_weight=w, eval_set=[(xv, yv)], verbose=False)
        p2 = m2.predict(xv); t2 = m2.predict(X_test)
        
        # 3. CatBoost Reg
        tp = Pool(xt, yt, weight=w); vp = Pool(xv, yv, weight=wt[val])
        m3 = CatBoostRegressor(**BP_CAT)
        m3.fit(tp, eval_set=vp, early_stopping_rounds=50)
        p3 = m3.predict(xv); t3 = m3.predict(X_test)

        # 4. LGBM Classifier (Fixed Robust)
        yc_t = y_class[tr]; yc_v = y_class[val]
        unseen = set(np.unique(yc_v)) - set(np.unique(yc_t))
        m4 = lgb.LGBMClassifier(**LGB_CLS_PARAMS)
        if len(unseen)>0: m4.fit(xt, yc_t, sample_weight=w)
        else: m4.fit(xt, yc_t, sample_weight=w, eval_set=[(xv, yc_v)], callbacks=[lgb.early_stopping(50, verbose=False)])
        
        p4_prob = m4.predict_proba(xv)
        if p4_prob.shape[1]!=11: 
            z = np.zeros((len(xv),11)); z[:, m4.classes_] = p4_prob; p4_prob = z
        p4 = np.sum(p4_prob * classes, axis=1)
        
        tp4_prob = m4.predict_proba(X_test)
        if tp4_prob.shape[1]!=11: 
            z = np.zeros((len(X_test),11)); z[:, m4.classes_] = tp4_prob; tp4_prob = z
        t4 = np.sum(tp4_prob * classes, axis=1)

        oof_expert[val, d] = (p1 + p2 + p3 + p4) / 4.0
        fold_test += (t1 + t2 + t3 + t4) / 4.0 / KFOLD
        del m1, m2, m3, m4; gc.collect()
    test_expert[:, d] = fold_test
    
    if USE_TEMP_CALIBRATION:
        try:
            iso = IsotonicRegression(out_of_bounds='clip').fit(oof_expert[:, d], y_full)
            setattr(sys.modules[__name__], f"iso_{d}", iso)
        except: pass


# STAGE 6: STACKING

memprint("Stacking")
oof_ev = oof_expert; test_ev = test_expert
for d in range(NUM_DOMAINS):
    iso = getattr(sys.modules[__name__], f"iso_{d}", None)
    if iso:
        try:
            oof_ev[:, d] = iso.predict(oof_ev[:, d])
            test_ev[:, d] = iso.predict(test_ev[:, d])
        except: pass

y_int = np.clip(np.round(y_full).astype(int), 0, 10)
strat_bins = np.digitize(y_int, bins=[3, 7])
meta_oof = np.zeros(len(oof_expert))
meta_test = np.zeros(len(test_expert))
skf = StratifiedKFold(n_splits=KFOLD, shuffle=True, random_state=SEED)

for tr, val in skf.split(oof_expert, strat_bins):
    mdl = Ridge(alpha=1.0, random_state=SEED)
    mdl.fit(oof_expert[tr], y_full[tr])
    meta_oof[val] = mdl.predict(oof_expert[val])
    meta_test += mdl.predict(test_expert) / KFOLD

memprint(f"Meta OOF RMSE: {mean_squared_error(y_full, meta_oof, squared=False):.4f}")
final_preds = np.clip(np.round(meta_test * 2) / 2, 0.0, 10.0)
sub = pd.DataFrame({"ID": test_ids, "score": final_preds})
sub.to_csv("submission_hetero_dual_nodiff.csv", index=False, float_format="%.1f")
memprint(f"Saved. Mean: {final_preds.mean():.3f}")